# Week 5 Assignment: A Simplified Transformer Encoder

## IMDB sentiment classification

This notebook expands a token-and-positional-embedding model with a simplified Transformer encoder block. The block contains multi-head self-attention, a fully connected feedforward network, two residual connections, and two LayerNormalization layers. A GlobalAveragePooling1D layer connects the encoder output to the binary classifier.

The model is implemented with the Keras Functional API, as required by the assignment.

## 1. Load and prepare the IMDB data

Keras provides IMDB reviews as integer-indexed sequences. Each integer represents a word in the IMDB vocabulary. The original training set is split into training and validation sets, while the official test set remains untouched until the final evaluation.

All sequences are padded or truncated to a fixed length so that they can be processed in batches.

In [ ]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers, Model
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

vocab_size = 20000
max_length = 200
batch_size = 128

(x_full, y_full), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=vocab_size
)

x_full = keras.utils.pad_sequences(
    x_full, maxlen=max_length, padding='post', truncating='post'
)
x_test = keras.utils.pad_sequences(
    x_test, maxlen=max_length, padding='post', truncating='post'
)
y_full = np.asarray(y_full).astype('float32')
y_test = np.asarray(y_test).astype('float32')

# Deterministic 80/20 split of the original training data.
rng = np.random.default_rng(42)
indices = rng.permutation(len(x_full))
x_full, y_full = x_full[indices], y_full[indices]

split = int(0.8 * len(x_full))
x_train, x_val = x_full[:split], x_full[split:]
y_train, y_val = y_full[:split], y_full[split:]

print('Training:', x_train.shape, y_train.shape)
print('Validation:', x_val.shape, y_val.shape)
print('Test:', x_test.shape, y_test.shape)
print('Sequence length:', max_length)
print('Vocabulary size used:', vocab_size)

## 2. Token and positional embedding

The token embedding converts each word index into a dense vector. Self-attention alone does not know the order of tokens, so a trainable positional embedding is added to the token representation. The output shape is `(batch, sequence_length, embedding_dim)`.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, vocabulary_size, sequence_length, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = layers.Embedding(vocabulary_size, embedding_dim)
        self.position_embedding = layers.Embedding(sequence_length, embedding_dim)
        self.sequence_length = sequence_length

    def call(self, token_ids):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        token_vectors = self.token_embedding(token_ids)
        position_vectors = self.position_embedding(positions)
        return token_vectors + position_vectors

    def get_config(self):
        config = super().get_config()
        config.update({'sequence_length': self.sequence_length})
        return config

## 3. Simplified Transformer encoder block

The block follows this structure:

```text
Token + positional embedding
              ↓
Multi-head self-attention
              ↓
Residual Add: attention input + attention output
              ↓
LayerNormalization
              ↓
Dense(ReLU) → Dense(linear)
              ↓
Residual Add: normalized representation + feedforward output
              ↓
LayerNormalization
              ↓
GlobalAveragePooling1D
              ↓
Sigmoid output
```

The attention and feedforward outputs have the same last dimension as the residual branches, namely `embedding_dim`. This makes the element-wise Add operations dimensionally valid.

In [ ]:
embedding_dim = 64
num_heads = 4
feedforward_dim = 128
dropout_rate = 0.20

inputs = layers.Input(shape=(max_length,), dtype='int32', name='token_ids')
embedding_output = TokenAndPositionEmbedding(
    vocab_size, max_length, embedding_dim, name='token_position_embedding'
)(inputs)

attention_output = layers.MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embedding_dim // num_heads,
    dropout=dropout_rate,
    name='multi_head_self_attention'
)(embedding_output, embedding_output)
attention_output = layers.Dropout(dropout_rate)(attention_output)

# First residual connection and normalization.
attention_residual = layers.Add(name='attention_residual')([
    embedding_output, attention_output
])
attention_normalized = layers.LayerNormalization(
    epsilon=1e-6, name='attention_layer_normalization'
)(attention_residual)

# Position-wise fully connected feedforward network.
feedforward_output = layers.Dense(
    feedforward_dim, activation='relu', name='feedforward_relu'
)(attention_normalized)
feedforward_output = layers.Dense(
    embedding_dim, activation=None, name='feedforward_linear'
)(feedforward_output)
feedforward_output = layers.Dropout(dropout_rate)(feedforward_output)

# Second residual connection and normalization.
feedforward_residual = layers.Add(name='feedforward_residual')([
    attention_normalized, feedforward_output
])
encoder_output = layers.LayerNormalization(
    epsilon=1e-6, name='feedforward_layer_normalization'
)(feedforward_residual)

pooled_output = layers.GlobalAveragePooling1D(name='global_average_pooling')(
    encoder_output
)
pooled_output = layers.Dropout(dropout_rate)(pooled_output)
outputs = layers.Dense(1, activation='sigmoid', name='sentiment_output')(
    pooled_output
)

model = Model(inputs=inputs, outputs=outputs, name='simplified_transformer_encoder')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 4. Why the tensor dimensions are compatible

With the selected hyperparameters:

- Embedding output: `(batch, 200, 64)`
- Multi-head attention output: `(batch, 200, 64)` because `4 × (64 / 4) = 64`
- First residual addition: `(batch, 200, 64) + (batch, 200, 64)`
- Feedforward hidden output: `(batch, 200, 128)`
- Feedforward final output: `(batch, 200, 64)`
- Second residual addition: `(batch, 200, 64) + (batch, 200, 64)`
- GlobalAveragePooling1D output: `(batch, 64)`
- Final sigmoid output: `(batch, 1)`

In [ ]:
# A forward pass confirms that the Functional graph is valid.
sample_output = model(x_train[:2])
print('Sample output shape:', sample_output.shape)
print('Sample probabilities:', sample_output.numpy().ravel())

## 5. Training

Binary cross-entropy is appropriate because the target has two classes and the final layer uses one sigmoid unit. Early stopping restores the weights from the epoch with the lowest validation loss.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=batch_size,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Training curves

The validation curves are used to monitor generalization and possible overfitting.

In [ ]:
accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(accuracy) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, accuracy, label='Training accuracy')
axes[0].plot(epochs, val_accuracy, label='Validation accuracy')
axes[0].set_title('Training and validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(epochs, loss, label='Training loss')
axes[1].plot(epochs, val_loss, label='Validation loss')
axes[1].set_title('Training and validation loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Binary cross-entropy')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Best validation accuracy: {max(val_accuracy):.4f}')

## 7. Final test evaluation

The test set is evaluated after model selection and training. It has not been used for gradient updates or hyperparameter selection.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, batch_size=batch_size, verbose=1)
print(f'Final test loss: {test_loss:.4f}')
print(f'Final test accuracy: {test_accuracy:.4f}')

## Conclusion

The final model is a simplified Transformer encoder for binary sentiment classification. It combines token embeddings and trainable positional embeddings with self-attention. The first residual connection stabilizes the attention sublayer, and the second residual connection stabilizes the feedforward sublayer. Layer normalization follows each residual addition. Global average pooling converts the sequence representation into one vector for the final sigmoid classifier.

The reported test accuracy is produced by the final evaluation cell after the notebook is executed.